# AquaSense AI – Stage 3: Classification Models

**Goal:** Use the same Maharashtra monthly rainfall dataset from Stage 1 and Stage 2 to classify the *following month's rainfall* into data-driven categories, then compare four classification algorithms.

We will implement:
1. Support Vector Machine (SVM)
2. Decision Tree (CART)
3. K-Nearest Neighbors (KNN)
4. Gaussian Naive Bayes

We will compare accuracy, precision, recall, F1-score, balanced accuracy, log loss, confusion matrices and multiclass ROC-AUC where it can be calculated.

> **Scope note:** This stage predicts rainfall categories, not water scarcity. The current dataset contains station-level rainfall features. A real water-scarcity classifier will require reliable supply, demand, reservoir, groundwater and consumption labels later.

## 0. How this notebook fits into AquaSense AI

- **Stage 1:** Understand the dataset and define the prediction target.
- **Stage 2:** Predict next-month rainfall as a numerical value using regression.
- **Stage 3 (current):** Predict whether next-month rainfall is Low, Moderate or High.

We reuse the same raw/feature-engineered CSV. We do not create fake observations.

## 1. Install and import the required libraries

We use `pandas` for tabular data, `matplotlib`/`seaborn` for plots, and `scikit-learn` for preprocessing, classifiers and evaluation metrics. The preprocessing steps are placed inside model pipelines so imputers, encoders and scalers are fitted on training data only.

In [ ]:
# Google Colab already includes the main libraries used in this notebook.
# If you are running locally and a library is missing, install it in a separate cell.

import io
import os
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, log_loss, confusion_matrix,
    classification_report, roc_auc_score, roc_curve, auc
)
import joblib

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RANDOM_STATE = 42
print("Libraries imported successfully.")

## 2. Upload and load the existing AquaSense rainfall CSV

Upload the **same monthly feature-engineered CSV** used in Stage 2. The reduced file is fine as long as it retains `station_key`, `year_month`, `observation_coverage`, the rainfall predictor columns and `next_month_rainfall_mm`. This cell also accepts a `.csv.gz` file.

In [ ]:
uploaded = {}
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Google Colab upload dialog is unavailable. Looking for a CSV in the current folder.")

if uploaded:
    filename = next(iter(uploaded.keys()))
    file_bytes = uploaded[filename]
    if filename.lower().endswith(".gz"):
        df = pd.read_csv(io.BytesIO(file_bytes), compression="gzip")
    else:
        df = pd.read_csv(io.BytesIO(file_bytes))
else:
    candidates = list(Path(".").glob("*.csv")) + list(Path(".").glob("*.csv.gz"))
    if not candidates:
        raise FileNotFoundError(
            "No CSV was uploaded or found. In Colab, rerun this cell and upload "
            "aquasense_rainfall_monthly_feature_engineered_reduced.csv."
        )
    filename = str(candidates[0])
    df = pd.read_csv(filename, compression="gzip" if filename.lower().endswith(".gz") else "infer")

print("Loaded file:", filename)
print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the dataset and validate required columns

Each row should represent one monitoring station in one month. The target `next_month_rainfall_mm` is the actual rainfall recorded in the *following consecutive calendar month* for the same station. We must not use the target itself, station ID, date string or any future information as a model input.

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Column names:")
print(df.columns.tolist())

TARGET = "next_month_rainfall_mm"
DATE_COLUMN = "year_month"
STATION_COLUMN = "station_key"

required_columns = [TARGET, DATE_COLUMN, STATION_COLUMN, "observation_coverage"]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}. "
        "Please upload the monthly feature-engineered AquaSense CSV created in Stage 1."
    )

print("\nRequired columns are present.")
display(df.dtypes.to_frame("data_type"))

## 4. Define the prediction problem

**Question:** Given rainfall and seasonal information observed up to the current station-month, can we classify rainfall in the next consecutive month?

- **Independent variables (X):** rainfall statistics and lag/seasonal information from the current or previous months.
- **Dependent variable (y):** `next_month_rainfall_mm`, converted into Low / Moderate / High classes.
- **Why classification?** The model output is a category rather than a continuous millimetre value.

We will derive category boundaries from the **training target values only**. This avoids using validation/test information to define the labels. The categories are relative to the training data; they are not official meteorological or water-scarcity standards.

## 5. Prepare and validate the time-series records

We keep only records with a valid station, month, target and at least 80% current-month observation coverage. We also verify that the next recorded row for that station is exactly the next calendar month and that its observation coverage is at least 80%. This matches the data-quality logic used in Stage 1/2.

In [ ]:
work = df.copy()
work[DATE_COLUMN] = pd.to_datetime(work[DATE_COLUMN], errors="coerce")
work[STATION_COLUMN] = work[STATION_COLUMN].astype("string").str.strip()

# Convert target and candidate numerical predictors to numeric values.
NUMERIC_FEATURE_CANDIDATES = [
    "rainfall_sum_observed_mm",
    "mean_daily_rainfall_mm",
    "median_daily_rainfall_mm",
    "max_daily_rainfall_mm",
    "std_daily_rainfall_mm",
    "observed_days",
    "rainy_days",
    "heavy_rain_days_20mm",
    "very_heavy_rain_days_50mm",
    "observation_coverage",
    "rainy_day_fraction",
    "heavy_rain_day_fraction",
    "rainfall_intensity_mm_per_rainy_day",
    "month_sin",
    "month_cos",
    "rainfall_lag_1m",
    "rainfall_prev3m_mean",
    "rainfall_prev3m_std",
    "prior_same_month_mean_mm",
    "rainfall_anomaly_vs_prior_climatology",
]
CATEGORICAL_FEATURE_CANDIDATES = ["season", "district"]

for col in list(set(NUMERIC_FEATURE_CANDIDATES + [TARGET, "observation_coverage"])):
    if col in work.columns:
        work[col] = pd.to_numeric(work[col], errors="coerce")

work = work.sort_values([STATION_COLUMN, DATE_COLUMN]).reset_index(drop=True)
groups = work.groupby(STATION_COLUMN, sort=False)
work["_following_record_month"] = groups[DATE_COLUMN].shift(-1)
work["_following_record_coverage"] = groups["observation_coverage"].shift(-1)

current_period = work[DATE_COLUMN].dt.to_period("M")
following_period = work["_following_record_month"].dt.to_period("M")
work["_is_next_calendar_month"] = (
    following_period.astype("int64") - current_period.astype("int64") == 1
)

MIN_COVERAGE = 0.80
valid_mask = (
    work[DATE_COLUMN].notna()
    & work[STATION_COLUMN].notna()
    & work["observation_coverage"].ge(MIN_COVERAGE)
    & work["_following_record_coverage"].ge(MIN_COVERAGE)
    & work["_is_next_calendar_month"]
    & work[TARGET].notna()
)

model_data = work.loc[valid_mask].copy()
model_data = model_data.drop_duplicates([STATION_COLUMN, DATE_COLUMN], keep="first")
model_data = model_data.sort_values([DATE_COLUMN, STATION_COLUMN]).reset_index(drop=True)

print("Rows in uploaded file:", len(df))
print("Rows after quality and next-month checks:", len(model_data))
print("Stations retained:", model_data[STATION_COLUMN].nunique())
print("Unique input months:", model_data[DATE_COLUMN].dt.to_period("M").nunique())
print("Date range:", model_data[DATE_COLUMN].min(), "to", model_data[DATE_COLUMN].max())

if len(model_data) < 60:
    raise ValueError("Fewer than 60 usable station-month rows remain. More valid source observations are needed.")
if model_data[DATE_COLUMN].dt.to_period("M").nunique() < 12:
    raise ValueError("At least 12 distinct months are recommended for the chronological train/validation/test split.")

display(model_data[[STATION_COLUMN, DATE_COLUMN, "observation_coverage", TARGET]].head())

## 6. Select input features (X) and target (y)

We use only the columns that exist in your uploaded CSV. We deliberately exclude `station_key`, `year_month`, the target itself and helper columns. The station identifier is not used as a predictor because the model could memorize station identity rather than learn general rainfall patterns. District and season can be used as categorical predictors if they are present.

In [ ]:
numeric_features = [c for c in NUMERIC_FEATURE_CANDIDATES if c in model_data.columns]
categorical_features = [c for c in CATEGORICAL_FEATURE_CANDIDATES if c in model_data.columns]
feature_columns = numeric_features + categorical_features

if len(numeric_features) == 0:
    raise ValueError("No expected numerical rainfall predictors were found in this CSV.")

print("Numerical input features:")
print(numeric_features)
print("\nCategorical input features:")
print(categorical_features)
print("\nTotal predictors:", len(feature_columns))
print("Target:", TARGET)

X_preview = model_data[feature_columns].copy()
y_preview = model_data[TARGET].copy()
print("\nX shape:", X_preview.shape)
print("y shape:", y_preview.shape)
display(X_preview.head())
display(y_preview.head().to_frame(TARGET))

## 7. Split data chronologically (70% train, 15% validation, 15% test)

Rainfall is time-dependent, so a random split could train on later months and test on earlier months. Instead, all stations from earlier months go into training, the next block of months goes into validation, and the latest block goes into testing.

- **Training:** fit the models and learn class thresholds.
- **Validation:** compare models and select a model.
- **Test:** final unbiased comparison after model selection.

The target is the next month, so we keep training target months before validation starts, and validation target months before testing starts.

In [ ]:
model_data["_period"] = model_data[DATE_COLUMN].dt.to_period("M")
model_data["_target_period"] = model_data["_period"] + 1
unique_periods = np.array(sorted(model_data["_period"].unique()))
n_periods = len(unique_periods)

train_boundary_index = int(n_periods * 0.70)
test_boundary_index = int(n_periods * 0.85)
train_boundary_index = max(2, min(train_boundary_index, n_periods - 4))
test_boundary_index = max(train_boundary_index + 2, min(test_boundary_index, n_periods - 2))
validation_start = unique_periods[train_boundary_index]
test_start = unique_periods[test_boundary_index]

train_df = model_data[model_data["_target_period"] < validation_start].copy()
validation_df = model_data[
    (model_data["_period"] >= validation_start)
    & (model_data["_target_period"] < test_start)
].copy()
test_df = model_data[model_data["_period"] >= test_start].copy()

print("Training input months:", train_df["_period"].min(), "to", train_df["_period"].max())
print("Validation input months:", validation_df["_period"].min(), "to", validation_df["_period"].max())
print("Testing input months:", test_df["_period"].min(), "to", test_df["_period"].max())
print("\nTraining rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

if min(len(train_df), len(validation_df), len(test_df)) == 0:
    raise ValueError("A chronological split created an empty partition. Check the data's date coverage.")

assert train_df["_target_period"].max() < validation_start
assert validation_df["_target_period"].max() < test_start
assert train_df["_period"].max() < validation_df["_period"].min()
assert validation_df["_period"].max() < test_df["_period"].min()
print("\nChronological split checks passed.")

## 8. Convert rainfall values into Low / Moderate / High classes

Our original target is numerical rainfall in millimetres. To create a classification problem, we need categories. We calculate two cut-points from the training target distribution only, aiming for approximately equal-sized classes. The same cut-points are then applied unchanged to validation and test targets.

This is a **relative classification**: Low, Moderate and High mean low/moderate/high relative to the training period. They are not official rainfall warnings or water-stress labels. If the training data has too many tied values to support three distinct classes, the code falls back to two classes and explains why.

In [ ]:
def fit_rainfall_class_bins(training_values):
    """Find data-driven cut-points using training targets only.

    Cut-points are placed between distinct observed values so all classes
    represented in training contain at least one training observation.
    """
    values = pd.to_numeric(pd.Series(training_values), errors="coerce").dropna().to_numpy()
    if len(values) < 3:
        raise ValueError("Not enough training target values to create rainfall classes.")

    unique_values, counts = np.unique(values, return_counts=True)
    if len(unique_values) < 2:
        raise ValueError("Training target has only one distinct rainfall value; classification is not possible.")

    if len(unique_values) >= 3:
        cumulative = np.cumsum(counts)
        n = len(values)
        first_candidates = np.arange(1, len(unique_values) - 1)
        first_idx = int(first_candidates[np.argmin(np.abs(cumulative[first_candidates - 1] - n / 3))])
        second_candidates = np.arange(first_idx + 1, len(unique_values))
        second_idx = int(second_candidates[np.argmin(np.abs(cumulative[second_candidates - 1] - 2 * n / 3))])

        cut1 = float(unique_values[first_idx - 1] + (unique_values[first_idx] - unique_values[first_idx - 1]) / 2)
        cut2 = float(unique_values[second_idx - 1] + (unique_values[second_idx] - unique_values[second_idx - 1]) / 2)
        return [-np.inf, cut1, cut2, np.inf], ["Low", "Moderate", "High"]
    else:
        # With only two distinct training values, use a binary classification task.
        cut = float(unique_values[0] + (unique_values[1] - unique_values[0]) / 2)
        return [-np.inf, cut, np.inf], ["Low", "High"]


def apply_rainfall_classes(values, bin_edges, class_names):
    return pd.cut(
        pd.to_numeric(values, errors="coerce"),
        bins=bin_edges,
        labels=class_names,
        include_lowest=True,
        right=True
    )

bin_edges, class_names = fit_rainfall_class_bins(train_df[TARGET])
print("Class names:", class_names)
print("Training-derived rainfall boundaries (mm):")
for i, name in enumerate(class_names):
    lower = bin_edges[i]
    upper = bin_edges[i + 1]
    print(f"{name}: ({lower:.3f}, {upper:.3f}] mm")

train_df["rainfall_class"] = apply_rainfall_classes(train_df[TARGET], bin_edges, class_names)
validation_df["rainfall_class"] = apply_rainfall_classes(validation_df[TARGET], bin_edges, class_names)
test_df["rainfall_class"] = apply_rainfall_classes(test_df[TARGET], bin_edges, class_names)

for split_name, split_df in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"\n{split_name} class counts:")
    display(split_df["rainfall_class"].value_counts().reindex(class_names, fill_value=0).to_frame("records"))

if train_df["rainfall_class"].isna().any():
    raise ValueError("Some training targets could not be assigned to a rainfall class.")
if train_df["rainfall_class"].nunique() < 2:
    raise ValueError("Training data contains fewer than two classes; cannot train classifiers.")

# Encoded integer labels make metrics and plots easier to compute consistently.
class_to_id = {name: idx for idx, name in enumerate(class_names)}
id_to_class = {idx: name for name, idx in class_to_id.items()}
for split_df in [train_df, validation_df, test_df]:
    split_df["class_id"] = split_df["rainfall_class"].map(class_to_id).astype("Int64")

# Drop any validation/test rows whose rainfall lies outside any defined class only if invalid.
train_df = train_df.dropna(subset=["class_id"]).copy()
validation_df = validation_df.dropna(subset=["class_id"]).copy()
test_df = test_df.dropna(subset=["class_id"]).copy()
train_df["class_id"] = train_df["class_id"].astype(int)
validation_df["class_id"] = validation_df["class_id"].astype(int)
test_df["class_id"] = test_df["class_id"].astype(int)

print("\nClass encoding:", class_to_id)

## 9. Prepare X and y for each split

All four models receive the same feature columns and the same train/validation/test observations. This is necessary for a fair comparison. We do not include the date, station key or target rainfall as an input.

In [ ]:
X_train = train_df[feature_columns].copy()
y_train = train_df["class_id"].copy()
X_val = validation_df[feature_columns].copy()
y_val = validation_df["class_id"].copy()
X_test = test_df[feature_columns].copy()
y_test = test_df["class_id"].copy()

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_val:  ", X_val.shape, " y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, " y_test: ", y_test.shape)
print("\nMissing numerical values in training predictors:")
display(X_train[numeric_features].isna().sum().to_frame("missing_count"))

## 10. Build preprocessing pipelines

- Numerical features: fill missing values with the training median, then standardize them. Scaling is especially important for SVM and KNN because they depend on distances/margins.
- Categorical features: fill missing values with `Unknown`, then one-hot encode them.
- The preprocessing is fitted inside each model pipeline, preventing information from validation/test data from leaking into training.

In [ ]:
# Dense one-hot output works with Gaussian Naive Bayes and is compatible with all four models.
try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn versions
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

transformers = []
if numeric_features:
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    transformers.append(("numeric", numeric_pipeline, numeric_features))

if categorical_features:
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", one_hot)
    ])
    transformers.append(("categorical", categorical_pipeline, categorical_features))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

print("Preprocessing pipeline ready.")

## 11. Define the four classification models

1. **SVM:** Finds a decision boundary that separates rainfall classes. We use an RBF kernel and probability estimates for log loss/ROC-AUC.
2. **Decision Tree (CART):** Learns a sequence of feature-based rules. We limit its depth and minimum leaf size to reduce overfitting.
3. **KNN:** Classifies an observation based on nearby training observations. Scaling is important because KNN uses distance.
4. **Gaussian Naive Bayes:** Uses Bayes’ theorem with a conditional-independence assumption and a Gaussian distribution for numerical inputs.

A majority-class baseline is also included. A model should outperform this simple baseline to be useful.

In [ ]:
n_classes = len(class_names)
if y_train.nunique() < 2:
    raise ValueError("The training split has fewer than two classes.")

model_specs = {
    "SVM": SVC(
        kernel="rbf", C=1.0, gamma="scale", probability=True,
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree (CART)": DecisionTreeClassifier(
        criterion="gini", max_depth=8, min_samples_leaf=4,
        class_weight="balanced", random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=min(7, len(X_train)), weights="distance", metric="minkowski"
    ),
    "Gaussian Naive Bayes": GaussianNB()
}

models = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])
    for name, estimator in model_specs.items()
}

baseline_model = DummyClassifier(strategy="most_frequent")
print("Models ready:", list(models.keys()))

## 12. Train the baseline and the four classifiers

We fit each model on the training block and predict the validation block. The validation block is used to compare/select models; it is not used to fit the initial models.

In [ ]:
baseline_model.fit(X_train, y_train)
validation_predictions = {
    "Majority-class baseline": baseline_model.predict(X_val)
}
validation_probabilities = {}

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    validation_predictions[name] = model.predict(X_val)
    validation_probabilities[name] = model.predict_proba(X_val)

print("\nAll models trained and validation predictions generated.")

## 13. Define evaluation metrics

- **Accuracy:** fraction of all predictions that are correct.
- **Precision (macro):** average precision across classes, giving each class equal weight.
- **Recall (macro):** average recall across classes.
- **F1 (macro):** balances precision and recall across classes; useful when class counts are uneven.
- **F1 (weighted):** averages class F1 using each class frequency.
- **Balanced accuracy:** average recall across classes.
- **Log loss:** evaluates predicted probabilities; lower is better.
- **ROC-AUC (OvR macro):** measures class discrimination using one-vs-rest ROC curves; higher is better.

We will select models using **validation macro-F1**, not test accuracy. The test set remains unseen until final evaluation.

In [ ]:
def calculate_classification_metrics(y_true, y_pred, y_probability, class_ids):
    result = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, labels=class_ids, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, labels=class_ids, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_true, y_pred, labels=class_ids, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_true, y_pred, labels=class_ids, average="weighted", zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
    }
    try:
        result["Log loss"] = log_loss(y_true, y_probability, labels=class_ids)
    except Exception:
        result["Log loss"] = np.nan

    # Macro one-vs-rest ROC-AUC requires every class to be represented in y_true.
    try:
        if set(np.unique(y_true)) == set(class_ids):
            result["ROC-AUC (OvR macro)"] = roc_auc_score(
                y_true, y_probability, labels=class_ids,
                multi_class="ovr", average="macro"
            )
        else:
            result["ROC-AUC (OvR macro)"] = np.nan
    except Exception:
        result["ROC-AUC (OvR macro)"] = np.nan
    return result

class_ids = list(range(len(class_names)))
validation_rows = []

# Baseline has no probability estimates in this comparison, so report core metrics only.
base_pred = validation_predictions["Majority-class baseline"]
validation_rows.append({
    "Model": "Majority-class baseline",
    "Accuracy": accuracy_score(y_val, base_pred),
    "Precision (macro)": precision_score(y_val, base_pred, labels=class_ids, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_val, base_pred, labels=class_ids, average="macro", zero_division=0),
    "F1 (macro)": f1_score(y_val, base_pred, labels=class_ids, average="macro", zero_division=0),
    "F1 (weighted)": f1_score(y_val, base_pred, labels=class_ids, average="weighted", zero_division=0),
    "Balanced accuracy": balanced_accuracy_score(y_val, base_pred),
    "Log loss": np.nan,
    "ROC-AUC (OvR macro)": np.nan,
})

for name in models:
    row = {"Model": name}
    row.update(calculate_classification_metrics(
        y_val, validation_predictions[name], validation_probabilities[name], class_ids
    ))
    validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["F1 (macro)", "Balanced accuracy", "Accuracy"], ascending=False
).reset_index(drop=True)

display(validation_results.round(4))

## 14. Compare validation performance

Macro-F1 is the primary selection metric because it treats each rainfall class equally. Accuracy is useful but can be misleading if one class dominates. Review the other metrics before choosing a model.

In [ ]:
plot_results = validation_results[validation_results["Model"] != "Majority-class baseline"].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(plot_results["Model"], plot_results["F1 (macro)"])
axes[0].set_title("Validation Macro-F1 (higher is better)")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(plot_results["Model"], plot_results["Accuracy"])
axes[1].set_title("Validation Accuracy (higher is better)")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

best_model_name = plot_results.sort_values(
    ["F1 (macro)", "Balanced accuracy", "Accuracy"], ascending=False
).iloc[0]["Model"]
print("Selected using validation Macro-F1:", best_model_name)

## 15. Refit all models on training + validation, then evaluate once on the test set

After selecting the model using validation data, we can use both training and validation blocks to fit the final version. The test block remains untouched until this point. We then report test metrics for all models for comparison. The class boundaries remain the ones learned from the original training block.

In [ ]:
train_validation_df = pd.concat([train_df, validation_df], axis=0).sort_values(
    [DATE_COLUMN, STATION_COLUMN]
).reset_index(drop=True)

X_trainval = train_validation_df[feature_columns].copy()
y_trainval = train_validation_df["class_id"].astype(int).copy()

final_models = {}
test_predictions = {}
test_probabilities = {}
test_rows = []

for name, estimator in model_specs.items():
    final_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])
    final_model.fit(X_trainval, y_trainval)
    pred = final_model.predict(X_test)
    proba = final_model.predict_proba(X_test)

    final_models[name] = final_model
    test_predictions[name] = pred
    test_probabilities[name] = proba

    row = {"Model": name}
    row.update(calculate_classification_metrics(y_test, pred, proba, class_ids))
    test_rows.append(row)

test_results = pd.DataFrame(test_rows).sort_values(
    ["F1 (macro)", "Balanced accuracy", "Accuracy"], ascending=False
).reset_index(drop=True)

display(test_results.round(4))
print("\nModel chosen from validation:", best_model_name)
print("Test-set performance of chosen model:")
display(test_results[test_results["Model"] == best_model_name].round(4))

## 16. Confusion matrix and classification report for the selected model

The confusion matrix shows which rainfall categories are confused with one another. Rows are actual categories and columns are predicted categories. The classification report provides precision, recall and F1 for each class.

In [ ]:
best_model = final_models[best_model_name]
best_test_predictions = test_predictions[best_model_name]
best_test_probabilities = test_probabilities[best_model_name]

cm = confusion_matrix(y_test, best_test_predictions, labels=class_ids)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names
)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title(f"Test Confusion Matrix – {best_model_name}")
plt.tight_layout()
plt.show()

print(classification_report(
    y_test, best_test_predictions,
    labels=class_ids, target_names=class_names, zero_division=0
))

## 17. Plot multiclass ROC curves for the selected model

For a multiclass problem, we draw one-vs-rest ROC curves: one class is treated as positive and all remaining classes are treated as negative. If the test data does not contain both positive and negative examples for a class, its ROC curve cannot be estimated and the notebook will say so.

In [ ]:
plt.figure(figsize=(8, 6))
plotted_any = False

for class_id, class_name in enumerate(class_names):
    y_binary = (y_test.to_numpy() == class_id).astype(int)
    if len(np.unique(y_binary)) < 2:
        print(f"ROC curve unavailable for {class_name}: test set has only one outcome for this class.")
        continue

    fpr, tpr, _ = roc_curve(y_binary, best_test_probabilities[:, class_id])
    class_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} (AUC = {class_auc:.3f})")
    plotted_any = True

if plotted_any:
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"One-vs-Rest ROC Curves – {best_model_name}")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("No ROC curves could be plotted because the test set lacks class variation.")

## 18. Understand model errors and feature importance

A model with the highest accuracy is not automatically the best. Check class-wise recall and the confusion matrix to see whether it systematically misses High rainfall months. For the Decision Tree, we can inspect feature importance as an interpretable diagnostic. These importances describe the fitted tree; they are not proof of causation.

In [ ]:
tree_model = final_models["Decision Tree (CART)"]
tree_preprocessor = tree_model.named_steps["preprocessor"]
tree_classifier = tree_model.named_steps["classifier"]

try:
    transformed_feature_names = tree_preprocessor.get_feature_names_out()
    importance_df = pd.DataFrame({
        "Feature": transformed_feature_names,
        "Importance": tree_classifier.feature_importances_
    }).sort_values("Importance", ascending=False)
    display(importance_df.head(15))

    top_features = importance_df.head(15).sort_values("Importance", ascending=True)
    plt.figure(figsize=(9, 5))
    plt.barh(top_features["Feature"], top_features["Importance"])
    plt.title("Decision Tree – Top Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Feature importance display was not available:", exc)

## 19. Make a prediction for one test observation

This shows how a trained model is used after training. We take one real row from the test set, ask the selected model to predict its category, and compare that prediction with the known next-month rainfall category.

In [ ]:
sample_X = X_test.iloc[[0]]
sample_actual_id = int(y_test.iloc[0])
sample_predicted_id = int(best_model.predict(sample_X)[0])
sample_probability = best_model.predict_proba(sample_X)[0]

print("Actual category:", id_to_class[sample_actual_id])
print("Predicted category:", id_to_class[sample_predicted_id])
print("Predicted probabilities:")
for class_id, probability in enumerate(sample_probability):
    print(f"  {id_to_class[class_id]}: {probability:.3f}")

display(sample_X)

## 20. Save the trained models and comparison results

We save the full pipelines, not just the classifiers. This is important because the imputer, scaler and one-hot encoder must be applied in exactly the same way when the model is used later in the Gradio application.

In [ ]:
ARTIFACT_DIR = Path("aquasense_classification_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

for name, model in final_models.items():
    safe_name = (
        name.lower().replace(" ", "_").replace("(", "").replace(")", "")
        .replace("-", "_")
    )
    joblib.dump(model, ARTIFACT_DIR / f"{safe_name}.joblib")

validation_results.to_csv(ARTIFACT_DIR / "validation_model_comparison.csv", index=False)
test_results.to_csv(ARTIFACT_DIR / "test_model_comparison.csv", index=False)

metadata = {
    "target_column": TARGET,
    "date_column": DATE_COLUMN,
    "station_column": STATION_COLUMN,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "feature_columns": feature_columns,
    "class_names": class_names,
    "class_to_id": class_to_id,
    "training_derived_bin_edges": [float(x) if np.isfinite(x) else str(x) for x in bin_edges],
    "selected_model_by_validation_macro_f1": best_model_name,
    "random_state": RANDOM_STATE,
    "note": "Rainfall categories are relative to training target distribution; not official rainfall warning or water scarcity classes."
}
with open(ARTIFACT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Save actual test predictions for later analysis/reporting.
prediction_export = test_df[[STATION_COLUMN, DATE_COLUMN, TARGET, "rainfall_class", "class_id"]].copy()
prediction_export["predicted_class_id"] = best_test_predictions
prediction_export["predicted_class"] = [id_to_class[int(x)] for x in best_test_predictions]
prediction_export.to_csv(ARTIFACT_DIR / "selected_model_test_predictions.csv", index=False)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
print("Files:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print("-", path.name, f"({path.stat().st_size / 1024:.1f} KB)")

## 21. Download the model artifacts

The ZIP contains the trained model pipelines, model comparison tables, metadata and selected-model test predictions. Keep this ZIP for the later Gradio/Hugging Face integration.

In [ ]:
shutil.make_archive("aquasense_classification_artifacts", "zip", ARTIFACT_DIR)
print("Created aquasense_classification_artifacts.zip")
try:
    from google.colab import files
    files.download("aquasense_classification_artifacts.zip")
except ImportError:
    print("ZIP saved in the current working directory.")